# 07_micro_R6_R9

Focused, single-seed micro mechanism experiment. Edit the CONFIG cell and run all cells on Kaggle GPU.

In [ ]:
# Edit paths here. Both notebooks use the same frozen 00d resource bundle.
RESOURCE_ROOT='/kaggle/input/datasets/maithanhphuong/full-resources-v3/minifasnet_binary_crossdomain_resources_v2_lcc_pooled'
CELEBA_ROOT='/kaggle/input/datasets/attentionlayer241/celeba-spoof-for-face-antispoofing/CelebA_Spoof_/CelebA_Spoof'
LCC_ROOT='/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD'
MINIFASNET_SOURCE_PATH='/kaggle/input/datasets/maithanhphuong/minifasnet/MiniFASNet.py'
MINIFASNET_CHECKPOINT_PATH='/kaggle/input/datasets/maithanhphuong/minifasnet/2.7_80x80_MiniFASNetV2.pth'
OUTPUT_ROOT='/kaggle/working/focused_micro_R6_R9_v1'
RUN_FILTER=None  # None = all variants in this notebook; or a list of exact run IDs.
EVAL_CELEBA_TEST = False  # Optional full Test inference; disabled for micro screening.
RESUME=True
NUM_WORKERS=4
AMP=True
TRAIN_SEED=100
BATCH_SIZE=256
MAX_EPOCHS=20
MILESTONES=[6,14]
GAMMA=.2
EARLIEST_EARLY_STOP_EPOCH=8
EARLY_STOP_PATIENCE=4
WARMUP_EPOCHS=1
CSMR_BAND_CENTERS=[.15,.45,.75]
CSMR_BAND_SIGMA=.10
CSMR_PRESERVE_DC=True
SELECTION_CHUNK_SIZE=32
CROP_SCALE=2.7
INPUT_SIZE=80
METRIC_TIE_EPS=1e-12


In [ ]:
import importlib.util, inspect, json, math, os, random, zipfile
from collections import Counter, OrderedDict
from pathlib import Path
import albumentations as A
import cv2, numpy as np, pandas as pd, torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, roc_curve
from torch.utils.data import Dataset, DataLoader
from IPython.display import FileLink, display

AUG_SEED=TRAIN_SEED+300000
BBOX_SEED=TRAIN_SEED+500000
GAIN_SEED=TRAIN_SEED+11000
RANDOM_GAIN_SEED=TRAIN_SEED+11500
BAND_SEED=TRAIN_SEED+12000
GAIN_RANGE=(.65,.90)
SPLITS=('training','development','evaluation')
EXPECTED_LCC={'training':8299,'development':2948,'evaluation':7580,'combined':18827}
if RUN_FILTER is not None and (not isinstance(RUN_FILTER,(list,tuple)) or not RUN_FILTER or len(set(RUN_FILTER))!=len(RUN_FILTER) or set(RUN_FILTER)-set(RUNS)):
    raise ValueError('RUN_FILTER contains an unknown or duplicate run ID')
if NUM_WORKERS<1:raise ValueError('NUM_WORKERS must be >= 1')
if (BATCH_SIZE,MAX_EPOCHS,MILESTONES,GAMMA,TRAIN_SEED)!=(256,20,[6,14],.2,100):raise RuntimeError('Frozen micro recipe changed')
root=Path(RESOURCE_ROOT);out=Path(OUTPUT_ROOT);source=Path(MINIFASNET_SOURCE_PATH);pretrained=Path(MINIFASNET_CHECKPOINT_PATH)
for path in (source,pretrained,root/'manifests/mini_train10k.csv',root/'manifests/mini_val3k.csv',
             root/'manifests/crossdomain_celeba_source_dev_full.csv',root/'bbox/celeba_bbox_cache_full.json',
             root/'labels/celeba_auxiliary_labels.csv'):
    if not path.is_file():raise FileNotFoundError(path)
def read_json(path):return json.loads(Path(path).read_text())
def write_json(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(value,indent=2,sort_keys=True,allow_nan=False)+'\n')
def save_torch(path,value):
    path=Path(path);tmp=path.with_suffix(path.suffix+'.tmp')
    torch.save(value,tmp);os.replace(tmp,path)
def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False

RUNS=OrderedDict([
 ('R6_all3_ce25',{'method':'ALL3','clean_ce_weight':.75,'spectral_ce_weight':.25,'consistency_beta':0.0}),
 ('R7_harmful_gated_worst',{'method':'HARMFUL','clean_ce_weight':.75,'spectral_ce_weight':.25,'consistency_beta':0.0}),
 ('R8_worst_random_mix',{'method':'MIX','clean_ce_weight':.75,'spectral_ce_weight':.25,'consistency_beta':0.0}),
 ('R9_worst_ce25_kl002',{'method':'WORST_KL','clean_ce_weight':.75,'spectral_ce_weight':.25,'consistency_beta':.02}),
])
def require_record(rec,key):
    if rec.get('status')!='VALID' or rec.get('usable_for_pad') is not True or rec.get('bbox_origin')!='SCRFD' or rec.get('source_key')!=key or not str(rec.get('bbox_source','')).startswith('SCRFD'):raise RuntimeError('Unscorable/non-SCRFD: '+key)
    box=rec.get('bbox_xyxy')
    if box is None or len(box)!=4 or 'crop_factor' in rec:raise RuntimeError('Not a raw bbox: '+key)
    x1,y1,x2,y2=map(float,box)
    if not np.isfinite(box).all() or x2<=x1 or y2<=y1:raise RuntimeError('Invalid bbox: '+key)
    return rec

def mini_crop_bgr(image,bbox_xyxy,scale=CROP_SCALE,size=INPUT_SIZE):
    # Exact 03b CropImage-compatible geometry.
    src_h,src_w=image.shape[:2];x1,y1,x2,y2=map(float,bbox_xyxy);box_w,box_h=x2-x1,y2-y1
    if box_w<=0 or box_h<=0 or src_w<2 or src_h<2:raise ValueError('Invalid crop geometry')
    used=min((src_h-1)/box_h,(src_w-1)/box_w,float(scale));new_w,new_h=box_w*used,box_h*used
    cx,cy=box_w/2+x1,box_h/2+y1;left,top=cx-new_w/2,cy-new_h/2;right,bottom=cx+new_w/2,cy+new_h/2
    if left<0:right-=left;left=0
    if top<0:bottom-=top;top=0
    if right>src_w-1:left-=right-src_w+1;right=src_w-1
    if bottom>src_h-1:top-=bottom-src_h+1;bottom=src_h-1
    lx,ty,rx,by=map(int,(left,top,right,bottom))
    if lx<0 or ty<0 or rx>=src_w or by>=src_h or rx<lx or by<ty:raise RuntimeError('Crop escaped image')
    patch=image[ty:by+1,lx:rx+1]
    if patch.size==0:raise RuntimeError('Empty crop')
    return cv2.resize(patch,(size,size))

# The literature's augmentation ranges are fixed. Support both documented Albumentations APIs.
# ISONoise works in RGB/HSV; convert only within augmentation, then restore BGR for MiniFASNet.
iso_params=inspect.signature(A.ISONoise).parameters
brightness_params=inspect.signature(A.RandomBrightnessContrast).parameters
blur_params=inspect.signature(A.MotionBlur).parameters
if {'color_shift','intensity'}<=set(iso_params) and {'brightness_limit','contrast_limit'}<=set(brightness_params) and 'blur_limit' in blur_params:
    AUGMENTATION_API='legacy_limits'
    iso=A.ISONoise(color_shift=(.15,.35),intensity=(.2,.5),p=.2)
    bright=A.RandomBrightnessContrast(brightness_limit=.2,contrast_limit=.2,brightness_by_max=True,p=.3)
    motion=A.MotionBlur(blur_limit=5,p=.2)
elif {'color_shift_range','intensity_range'}<=set(iso_params) and {'brightness_range','contrast_range'}<=set(brightness_params) and 'blur_range' in blur_params:
    AUGMENTATION_API='range_parameters'
    iso=A.ISONoise(color_shift_range=(.15,.35),intensity_range=(.2,.5),p=.2)
    bright=A.RandomBrightnessContrast(brightness_range=(-.2,.2),contrast_range=(-.2,.2),brightness_by_max=True,p=.3)
    motion=A.MotionBlur(blur_range=(3,5),p=.2)
else:raise RuntimeError('Unsupported Albumentations augmentation API; freeze and audit a compatible version')
COPY_AUG=A.Compose([A.HorizontalFlip(p=.5),iso,bright,motion])
def copied_aug_bgr(patch):
    rgb=cv2.cvtColor(patch,cv2.COLOR_BGR2RGB)
    augmented=COPY_AUG(image=rgb)['image']
    return cv2.cvtColor(augmented,cv2.COLOR_RGB2BGR)
def tensor_of(patch):
    if patch.shape!=(80,80,3) or patch.dtype!=np.uint8:raise ValueError('Expected BGR uint8 80x80')
    return torch.from_numpy(np.ascontiguousarray(patch.transpose(2,0,1))).float().div_(255.)

class PADNet(nn.Module):
    def __init__(self,copied):
        super().__init__();self.copied=copied
        self.trunk=module.MiniFASNetV2(embedding_size=128,conv6_kernel=(5,5),drop_p=.2,num_classes=3,img_channel=3)
        if copied:
            self.trunk.prob=nn.Identity()
            self.pad_head=nn.Linear(128,2);self.spoof_type_head=nn.Linear(128,11)
            self.lighting_head=nn.Linear(128,5);self.attributes_head=nn.Linear(128,40)
    def features(self,x):
        t=self.trunk
        for layer in (t.conv1,t.conv2_dw,t.conv_23,t.conv_3,t.conv_34,t.conv_4,t.conv_45,t.conv_5,t.conv_6_sep,t.conv_6_dw,t.conv_6_flatten):x=layer(x)
        if t.embedding_size!=512:x=t.linear(x)
        x=t.drop(t.bn(x))
        if x.ndim!=2 or x.shape[1]!=128:raise RuntimeError('MiniFAS representation is not [B,128]')
        return x
    def forward(self,x,aux=False):
        z=self.features(x)
        if not self.copied:return self.trunk.prob(z)
        pad=self.pad_head(z)
        if aux:return pad,self.spoof_type_head(z),self.lighting_head(z),self.attributes_head(z)
        return pad

def pad_score(logits):
    z=torch.as_tensor(logits)
    if z.shape[1]==3:return (z[:,0]-torch.logsumexp(z[:,1:],dim=1)).float().cpu().numpy()
    if z.shape[1]==2:return (z[:,0]-z[:,1]).float().cpu().numpy()
    raise ValueError('Expected two or three PAD logits')
def threshold_from_val(y3,scores):
    y=np.asarray(y3,dtype=np.int64);s=np.asarray(scores,dtype=np.float64)
    if len(y)!=len(s) or not len(y) or not np.isfinite(s).all() or set(np.unique(y))!={0,1,2}:raise ValueError('Val needs all three classes')
    order=np.argsort(s,kind='mergesort');v=s[order];real=(y[order]==0);nr=int(real.sum());na=len(y)-nr
    bound=np.r_[0,np.flatnonzero(np.diff(v))+1,len(v)];rp=np.r_[0,np.cumsum(real)];ap=np.r_[0,np.cumsum(~real)];pairs=[]
    for j,k in enumerate(bound):
        t=float(np.nextafter(v[0],-np.inf)) if j==0 else (float(np.nextafter(v[-1],np.inf)) if j==len(bound)-1 else float(v[k-1]/2+v[k]/2))
        pairs.append((.5*(rp[k]/nr+(na-ap[k])/na),abs(t),t))
    return min(pairs)[2]
def metrics(y3,scores,threshold,candidate_n):
    y=np.asarray(y3,dtype=int);s=np.asarray(scores,dtype=float);real=y==0
    if len(y)!=len(s) or set(real)!={False,True} or not np.isfinite(s).all():raise ValueError('Invalid metric population')
    fpr,tpr,thr=roc_curve(real.astype(int),s);eer_idx=int(np.argmin(abs(fpr-(1-tpr))))
    pred=s>=threshold;apcer=float(pred[~real].mean());bpcer=float((~pred[real]).mean())
    return {'auc':float(roc_auc_score(real.astype(int),s)),'eer':float(.5*(fpr[eer_idx]+1-tpr[eer_idx])),
      'tpr_at_fpr_1pct':float(np.interp(.01,fpr,tpr)),'apcer':apcer,'bpcer':bpcer,'acer':.5*(apcer+bpcer),
      'hter':.5*(apcer+bpcer),'binary_accuracy':float((pred==real).mean()),'locked_source_threshold':float(threshold),
      'candidate_n':int(candidate_n),'scored_n':len(y),'detector_coverage':len(y)/candidate_n}

def radial_masks(size=INPUT_SIZE,device=None):
    if device is None:device=globals()['device']
    # Same radial geometry as the tested 03b CSMR notebook.
    y=torch.arange(size,device=device,dtype=torch.float32)-size//2
    yy,xx=torch.meshgrid(y,y,indexing='ij')
    radius=torch.sqrt(xx.square()+yy.square())/math.sqrt(2*(size//2)**2)
    centers=torch.as_tensor(CSMR_BAND_CENTERS,dtype=torch.float32,device=device)
    masks=torch.exp(-((radius[None]-centers[:,None,None])**2)/(2*CSMR_BAND_SIGMA**2))
    if masks.shape!=(3,size,size) or not torch.isfinite(masks).all():raise RuntimeError('Invalid CSMR radial masks')
    return masks
def spectral_views(x,gains,masks):
    if x.ndim!=4 or x.shape[1:]!=(3,80,80) or not torch.isfinite(x).all():raise ValueError('CSMR input')
    x=x.float();gains=gains.float().reshape(-1)
    if len(gains)!=len(x) or not ((gains>=0)&(gains<=1)).all():raise ValueError('CSMR gain')
    spectrum=torch.fft.fftshift(torch.fft.fft2(x,norm='ortho'),dim=(-2,-1))
    fields=1-(1-gains[:,None,None,None])*masks[None]
    if CSMR_PRESERVE_DC:fields[:,:,40,40]=1
    raw=torch.fft.ifft2(torch.fft.ifftshift(spectrum[:,None]*fields[:,:,None],dim=(-2,-1)),norm='ortho').real
    clips=((raw<0)|(raw>1)).float().mean(dim=(2,3,4));views=raw.clamp(0,1)
    if not torch.isfinite(views).all():raise RuntimeError('Nonfinite CSMR views')
    return views,clips
def aligned_margin(logits,binary):
    d=logits[:,0].float()-logits[:,1].float()
    return torch.where(binary==0,d,-d)

def auxiliary_loss(outputs,binary,spoof,lighting,attrs):
    pad,spoof_logits,light_logits,attr_logits=outputs
    ls=F.cross_entropy(spoof_logits,spoof);ll=F.cross_entropy(light_logits,lighting);mask=binary==0
    la=F.binary_cross_entropy_with_logits(attr_logits[mask],attrs[mask]) if mask.any() else attr_logits.sum()*0
    return ls,ll,la
def rng_state():
    return {'python':random.getstate(),'numpy':np.random.get_state(),'torch':torch.get_rng_state(),'cuda':torch.cuda.get_rng_state_all()}
def restore_rng(r):
    random.setstate(r['python']);np.random.set_state(r['numpy']);torch.set_rng_state(r['torch']);torch.cuda.set_rng_state_all(r['cuda'])
def bbox_jitter(rec,rng):
    x1,y1,x2,y2=map(float,rec['bbox_xyxy']);w,h=x2-x1,y2-y1
    if rng.random()<.20:
        base=max(w,h);cx,cy=(x1+x2)/2,(y1+y2)/2;scale=rng.uniform(.95,1.05)
        cx+=rng.uniform(-.05,.05)*base;cy+=rng.uniform(-.05,.05)*base;w*=scale;h*=scale
        return [cx-w/2,cy-h/2,cx+w/2,cy+h/2]
    return [x1,y1,x2,y2]

def copied_aug_bgr_seeded(patch,seed):
    # Augmentation RNG cannot consume bbox or spectral RNG state.
    random.seed(seed);np.random.seed(seed)
    if hasattr(COPY_AUG,'set_random_seed'):COPY_AUG.set_random_seed(seed)
    return copied_aug_bgr(patch)

def load_frame(name):return pd.read_csv(root/'manifests'/(name+'.csv'),keep_default_na=False)
train=load_frame('mini_train10k');val=load_frame('mini_val3k');test=load_frame('crossdomain_celeba_source_dev_full')
if (len(train),len(val),len(test))!=(10000,3000,67170):raise RuntimeError('Unexpected frozen CelebA populations')
if train.sample_key.duplicated().any() or val.sample_key.duplicated().any() or set(train.sample_key)&set(val.sample_key):
    raise RuntimeError('Train/Val sample overlap')
if set(train.subject_id.astype(str))&set(val.subject_id.astype(str)):raise RuntimeError('Train/Val subject overlap')
for frame in (train,val):
    if not frame.usable_for_pad.astype(bool).all() or set(frame.bbox_origin)!={'SCRFD'} or set(frame.three_class_label)!={0,1,2}:
        raise RuntimeError('Frozen mini split is not fully usable')
records=read_json(root/'bbox/celeba_bbox_cache_full.json')['records']
aux=pd.read_csv(root/'labels/celeba_auxiliary_labels.csv',keep_default_na=False).set_index('sample_key',verify_integrity=True)
for frame in (train,val,test):
    if not set(frame.sample_key)<=set(records) or not set(frame.sample_key)<=set(aux.index):raise RuntimeError('Missing CelebA cache/aux entry')
lcc_frames={};lcc_records={}
for split in ('training','development','evaluation','combined'):
    name='lcc_official_evaluation_full' if split=='evaluation' else 'lcc_'+split+'_full'
    cache_name='lcc_official_evaluation_bbox_cache' if split=='evaluation' else 'lcc_'+split+'_bbox_cache'
    path=root/'manifests'/(name+'.csv');cache=root/'bbox'/(cache_name+'.json')
    if not path.is_file() or not cache.is_file():raise FileNotFoundError((path,cache))
    frame=pd.read_csv(path,keep_default_na=False)
    if len(frame)!=EXPECTED_LCC[split] or frame.path.duplicated().any():raise RuntimeError('Unexpected LCC population: '+split)
    recs=read_json(cache)['records']
    if set(frame.path)!=set(recs):raise RuntimeError('LCC manifest/cache keys differ: '+split)
    lcc_frames[split]=frame;lcc_records[split]=recs
ordered_pool=pd.concat([lcc_frames[s].assign(lcc_split=s) for s in SPLITS],ignore_index=True)
if not lcc_frames['combined'].equals(ordered_pool):raise RuntimeError('Combined LCC manifest differs from physical splits')
if not torch.cuda.is_available():raise RuntimeError('Kaggle CUDA GPU required for the frozen micro batch size')
source_spec=importlib.util.spec_from_file_location('local_minifasnet_v2',source)
module=importlib.util.module_from_spec(source_spec);source_spec.loader.exec_module(module)
device=torch.device('cuda');use_amp=bool(AMP)
out.mkdir(parents=True,exist_ok=True);(out/'runs').mkdir(exist_ok=True)
print('GPU:',torch.cuda.get_device_name(0),'Train/Val:',len(train),len(val),'LCC:',{s:len(f) for s,f in lcc_frames.items()})

def init_model():
    seed_all(TRAIN_SEED)
    model=PADNet(True)
    raw=torch.load(pretrained,map_location='cpu',weights_only=True)
    if not isinstance(raw,OrderedDict) or not raw or not all(k.startswith('module.') for k in raw):
        raise RuntimeError('Official PAD checkpoint format differs')
    stripped=OrderedDict((k[7:],v) for k,v in raw.items())
    if tuple(stripped['prob.weight'].shape)!=(3,128) or tuple(stripped['conv_6_dw.conv.weight'].shape[-2:])!=(5,5):
        raise RuntimeError('Official PAD checkpoint architecture differs')
    compatible={k:v for k,v in stripped.items() if k!='prob.weight'}
    target=model.trunk.state_dict()
    if set(compatible)!=set(target) or any(tuple(compatible[k].shape)!=tuple(target[k].shape) for k in compatible):
        raise RuntimeError('PAD pretrained trunk keys differ')
    model.trunk.load_state_dict(compatible,strict=True)
    return model.to(device)

class CelebaData(Dataset):
    def __init__(self,frame,training=False,epoch=0):
        self.rows=frame.reset_index(drop=True);self.training=training;self.epoch=epoch
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        row=self.rows.iloc[i];key=str(row.sample_key);rec=require_record(records[key],key)
        image=cv2.imread(str(Path(CELEBA_ROOT)/key),cv2.IMREAD_COLOR)
        if image is None:raise FileNotFoundError(key)
        box=bbox_jitter(rec,random.Random(BBOX_SEED+self.epoch*100000+i)) if self.training else rec['bbox_xyxy']
        patch=mini_crop_bgr(image,box)
        if self.training:patch=copied_aug_bgr_seeded(patch,AUG_SEED+self.epoch*100000+i)
        a=aux.loc[key]
        return tensor_of(patch),int(row.three_class_label),int(row.binary_label),int(a.spoof_type),int(a.lighting),torch.tensor([float(a[f'attr_{j:02d}']) for j in range(40)]),key

class LCCData(Dataset):
    def __init__(self,split):
        self.split=split;self.rows=lcc_frames[split].loc[lcc_frames[split].usable_for_pad.astype(bool)].reset_index(drop=True)
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        row=self.rows.iloc[i];key=str(row.path);rec=require_record(lcc_records[self.split][key],key)
        path=(Path(LCC_ROOT)/'LCC_FASD_evaluation'/key) if self.split=='evaluation' else Path(LCC_ROOT)/key
        image=cv2.imread(str(path),cv2.IMREAD_COLOR)
        if image is None:raise FileNotFoundError(path)
        return tensor_of(mini_crop_bgr(image,rec['bbox_xyxy'])),int(row.label),key

def loader(ds,shuffle=False,epoch=0):
    gen=torch.Generator().manual_seed(TRAIN_SEED+epoch)
    return DataLoader(ds,batch_size=BATCH_SIZE,shuffle=shuffle,num_workers=NUM_WORKERS,pin_memory=True,
                      drop_last=False,generator=gen)

def infer(model,dl,lcc=False):
    model.eval();keys=[];ys=[];zs=[]
    with torch.no_grad():
        for batch in dl:
            if lcc:x,y,k=batch
            else:x,y,_,_,_,_,k=batch
            with torch.cuda.amp.autocast(enabled=use_amp):z=model(x.to(device,non_blocking=True))
            keys.extend(k);ys.extend(y.numpy().tolist());zs.append(z.float().cpu().numpy())
    if not zs:raise RuntimeError('No scorable samples')
    return list(map(str,keys)),np.asarray(ys,dtype=int),np.concatenate(zs)

def select_worst_view(model,base,binary,masks,gain_rng):
    was_training=model.training;model.eval()
    chosen=[];bands=[];gains_out=[];clips_out=[];clean_margins=[];worst_margins=[]
    try:
        with torch.no_grad():
            all_gains=torch.empty(len(base),device=base.device).uniform_(*GAIN_RANGE,generator=gain_rng)
            for start in range(0,len(base),SELECTION_CHUNK_SIZE):
                x=base[start:start+SELECTION_CHUNK_SIZE];y=binary[start:start+SELECTION_CHUNK_SIZE]
                gains=all_gains[start:start+len(x)]
                with torch.cuda.amp.autocast(enabled=use_amp):clean_z=model(x)
                clean_margins.append(aligned_margin(clean_z,y).detach())
                views,clips=spectral_views(x,gains,masks);margins=[]
                for band in range(3):
                    with torch.cuda.amp.autocast(enabled=use_amp):z=model(views[:,band])
                    margins.append(aligned_margin(z,y))
                margin=torch.stack(margins,dim=1);ix=margin.argmin(dim=1)
                row=torch.arange(len(x),device=x.device)
                chosen.append(views[row,ix].detach());bands.append(ix.detach())
                gains_out.append(gains.detach());clips_out.append(clips[row,ix].detach())
                worst_margins.append(margin[row,ix].detach())
    finally:model.train(was_training)
    return tuple(torch.cat(seq) for seq in (chosen,bands,gains_out,clips_out,clean_margins,worst_margins))

def select_random_view(base,masks,gain_rng,band_rng):
    with torch.no_grad():
        gains=torch.empty(len(base),device=base.device).uniform_(*GAIN_RANGE,generator=gain_rng)
        band=torch.randint(0,3,(len(base),),device=base.device,generator=band_rng)
        x=base.float();selected_masks=masks[band]
        spectrum=torch.fft.fftshift(torch.fft.fft2(x,norm='ortho'),dim=(-2,-1))
        field=1-(1-gains[:,None,None])*selected_masks
        if CSMR_PRESERVE_DC:field[:,40,40]=1
        raw=torch.fft.ifft2(torch.fft.ifftshift(spectrum*field[:,None],dim=(-2,-1)),norm='ortho').real
        return raw.clamp(0,1).detach(),band,gains

def training_terms(model,x,binary,spoof,lighting,attrs,spec,masks,gain_rng,random_gain_rng,band_rng,active):
    method=spec['method'];n=len(x);zero=x.new_zeros(())
    selected=None;bands=None;gains=None;select_clean=None;select_worst=None;random_bands=None
    if active and method in ('WORST','HARMFUL','MIX','WORST_KL'):
        selected,bands,gains,_,select_clean,select_worst=select_worst_view(model,x,binary,masks,gain_rng)
    elif active and method=='ALL3':
        gains=torch.empty(n,device=x.device).uniform_(*GAIN_RANGE,generator=gain_rng)
        with torch.no_grad():all_views,_=spectral_views(x,gains,masks)
    if active and method=='MIX':
        random_view,random_bands,random_gains=select_random_view(x,masks,random_gain_rng,band_rng)
    with torch.cuda.amp.autocast(enabled=use_amp):
        clean=model(x,aux=True);clean_ce=F.cross_entropy(clean[0],binary)
        ls,ll,la=auxiliary_loss(clean,binary,spoof,lighting,attrs)
        aux_total=.1*ls+.1*ll+la
        spectral_ce=clean_ce*0;kl=clean_ce*0;harmful_count=0
        spectral_logits=None;all3_logits=None
        if active and method in ('WORST','HARMFUL','MIX','WORST_KL'):
            spectral_logits=model(selected)
            ce_each=F.cross_entropy(spectral_logits,binary,reduction='none')
            if method=='HARMFUL':
                harmful=(select_worst<select_clean).detach()
                harmful_count=int(harmful.sum())
                spectral_ce=(ce_each*harmful).sum()/harmful_count if harmful_count else spectral_logits.sum()*0
            elif method=='MIX':
                random_logits=model(random_view)
                random_ce=F.cross_entropy(random_logits,binary)
                spectral_ce=.75*ce_each.mean()+.25*random_ce
            else:spectral_ce=ce_each.mean()
            if method=='WORST_KL':
                kl=F.kl_div(F.log_softmax(spectral_logits.float(),dim=1),
                    F.softmax(clean[0].float().detach(),dim=1),reduction='batchmean')
        elif active and method=='ALL3':
            all3_logits=[model(all_views[:,band]) for band in range(3)]
            spectral_ce=sum(F.cross_entropy(z,binary) for z in all3_logits)/3
        loss=(spec['clean_ce_weight']*clean_ce+spec['spectral_ce_weight']*spectral_ce+
              spec['consistency_beta']*kl+aux_total) if active else clean_ce+aux_total
    with torch.no_grad():
        clean_margin=aligned_margin(clean[0],binary).mean()
        worst_margin=select_worst.mean() if select_worst is not None else zero
        drop=(select_clean-select_worst).mean() if select_worst is not None else zero
        flips=int(((clean[0].argmax(dim=1)==binary)&(spectral_logits.argmax(dim=1)!=binary)).sum()) if spectral_logits is not None else 0
        if all3_logits is not None:
            flips=sum(int(((clean[0].argmax(dim=1)==binary)&(z.argmax(dim=1)!=binary)).sum()) for z in all3_logits)
    scalars={'train_total_loss':float(loss.detach()),'clean_ce':float(clean_ce.detach()),
        'spectral_ce':float(spectral_ce.detach()),'aux_loss':float(aux_total.detach()),
        'kl_loss':float(kl.detach()),'weighted_kl_contribution':float((spec['consistency_beta']*kl).detach()),
        'clean_aligned_margin':float(clean_margin.detach()),'worst_margin':float(worst_margin.detach()),
        'margin_drop':float(drop.detach()),'spectral_flip_rate':flips/(3*n if all3_logits is not None else n),
        'harmful_count':harmful_count,'harmful_fraction':harmful_count/n,
        'harmful_ce':float(spectral_ce.detach()) if method=='HARMFUL' and active else 0.0,
        'sampled_gain_mean':float(gains.mean()) if gains is not None else 0.0}
    counts=Counter(bands.detach().cpu().tolist()) if bands is not None else Counter()
    if active and method=='ALL3':counts=Counter({0:n,1:n,2:n})
    random_counts=Counter(random_bands.detach().cpu().tolist()) if random_bands is not None else Counter()
    return loss,scalars,counts,random_counts

def snapshot_rng(gain_rng,random_gain_rng,band_rng):
    return {'python':random.getstate(),'numpy':np.random.get_state(),'torch':torch.get_rng_state(),
            'cuda':torch.cuda.get_rng_state_all(),'gain':gain_rng.get_state(),
            'random_gain':random_gain_rng.get_state(),'band':band_rng.get_state()}
def restore_rng(state,gain_rng,random_gain_rng,band_rng):
    random.setstate(state['python']);np.random.set_state(state['numpy']);torch.set_rng_state(state['torch'])
    torch.cuda.set_rng_state_all(state['cuda']);gain_rng.set_state(state['gain'])
    random_gain_rng.set_state(state['random_gain']);band_rng.set_state(state['band'])

def run_config(name,spec):
    return {'run_id':name,'spec':spec,'resource_root':str(root),'train_manifest':'mini_train10k.csv',
      'val_manifest':'mini_val3k.csv','train_seed':TRAIN_SEED,'base_order':'torch Generator(TRAIN_SEED+epoch)',
      'bbox_rng_seed':BBOX_SEED,'augmentation_rng_seed':AUG_SEED,'gain_rng_seed':GAIN_SEED,
      'random_gain_rng_seed':RANDOM_GAIN_SEED,'random_band_rng_seed':BAND_SEED,
      'pretrained_checkpoint':str(pretrained),'model_source':str(source),
      'batch_size':BATCH_SIZE,'max_epochs':MAX_EPOCHS,'milestones':MILESTONES,'gamma':GAMMA,
      'optimizer':{'name':'SGD','lr':.005,'momentum':.9,'weight_decay':5e-4},
      'warmup_epochs':0 if spec['method']=='CLEAN' else WARMUP_EPOCHS,
      'earliest_early_stop_epoch':EARLIEST_EARLY_STOP_EPOCH,'patience':EARLY_STOP_PATIENCE,
      'spectral':{'centers':CSMR_BAND_CENTERS,'sigma':CSMR_BAND_SIGMA,'gain':[.65,.90],
                  'preserve_dc':CSMR_PRESERVE_DC},
      'input':'80x80 BGR float [0,1] 2.7x','augmentation_api':AUGMENTATION_API,
      'albumentations_version':A.__version__,'amp':use_amp}

def prediction_frame(frame,keys,y,z,threshold,keycol):
    scores=pad_score(z)
    scored=pd.DataFrame({keycol:keys,'pad_logit_score':scores,'pred_real':scores>=threshold,
                         'logit_0':z[:,0],'logit_1':z[:,1]})
    result=frame.merge(scored,on=keycol,how='left',validate='one_to_one',sort=False)
    valid=frame.usable_for_pad.astype(bool)
    if result.loc[valid,'pad_logit_score'].isna().any() or result.loc[~valid,'pad_logit_score'].notna().any():
        raise RuntimeError('Prediction accounting differs from frozen candidates')
    result['locked_threshold']=threshold
    return result

def evaluate_lcc(model,folder,locked):
    scored_predictions=[]
    for split in SPLITS:
        frame=lcc_frames[split]
        keys,y,z=infer(model,loader(LCCData(split)),True)
        expected=frame.loc[frame.usable_for_pad.astype(bool),'path'].astype(str).tolist()
        if keys!=expected:raise RuntimeError('LCC scored order differs: '+split)
        pred=prediction_frame(frame,keys,y,z,locked,'path').assign(lcc_split=split)
        mm=metrics(y,pad_score(z),locked,len(frame))
        write_json(folder/f'lcc_{split}_metrics.json',mm)
        pred.to_csv(folder/f'lcc_{split}_predictions.csv',index=False)
        scored_predictions.append(pred)
    pooled=pd.concat(scored_predictions,ignore_index=True)
    if not pooled[['path','label','status','usable_for_pad','bbox_origin','lcc_split']].equals(
        lcc_frames['combined'][['path','label','status','usable_for_pad','bbox_origin','lcc_split']]):
        raise RuntimeError('Combined prediction candidates differ from 00d manifest')
    usable=pooled.usable_for_pad.astype(bool)
    mm=metrics(pooled.loc[usable,'label'],pooled.loc[usable,'pad_logit_score'],locked,len(pooled))
    write_json(folder/'lcc_combined_metrics.json',mm)
    pooled.to_csv(folder/'lcc_combined_predictions.csv',index=False)

def evaluate_celeba_test(model,folder,locked):
    scored=test.loc[test.usable_for_pad.astype(bool)]
    keys,y,z=infer(model,loader(CelebaData(scored)))
    if keys!=scored.sample_key.astype(str).tolist():raise RuntimeError('CelebA Test order differs')
    write_json(folder/'test_metrics.json',metrics(y,pad_score(z),locked,len(test)))
    prediction_frame(test,keys,y,z,locked,'sample_key').to_csv(folder/'test_predictions.csv',index=False)

def run_one(name,spec):
    folder=out/'runs'/name;folder.mkdir(parents=True,exist_ok=True)
    cfg=run_config(name,spec);cfg_path=folder/'config.json'
    if cfg_path.is_file() and read_json(cfg_path)!=cfg:raise RuntimeError(name+' existing config differs')
    write_json(cfg_path,cfg)
    best_path=folder/'best.pth';last_path=folder/'last_state.pth';done_path=folder/'completion.json'
    if done_path.is_file():
        needed=[folder/(k+'_metrics.json') for k in ('val','lcc_training','lcc_development','lcc_evaluation','lcc_combined')]
        if not all(p.is_file() for p in needed) or not best_path.is_file():
            raise RuntimeError(name+' incomplete completion outputs')
        if EVAL_CELEBA_TEST and not all((folder/p).is_file() for p in ('test_metrics.json','test_predictions.csv')):
            freeze=read_json(folder/'evaluation_freeze.json')
            best=torch.load(best_path,map_location='cpu',weights_only=True)
            if best['run_id']!=name or best['config']!=cfg or best['epoch']!=freeze['best_epoch']:
                raise RuntimeError(name+' completed best checkpoint differs from Val freeze')
            model=PADNet(True).to(device);model.load_state_dict(best['state_dict']);model.eval()
            evaluate_celeba_test(model,folder,freeze['locked_threshold'])
            del model;torch.cuda.empty_cache()
            print(name,'CelebA Test evaluated from frozen checkpoint')
        else:print(name,'completed; skipping')
        return
    model=init_model()
    optimizer=torch.optim.SGD(model.parameters(),lr=.005,momentum=.9,weight_decay=5e-4)
    scheduler=torch.optim.lr_scheduler.MultiStepLR(optimizer,milestones=MILESTONES,gamma=GAMMA)
    scaler=torch.cuda.amp.GradScaler(enabled=use_amp)
    gain_rng=torch.Generator(device=device).manual_seed(GAIN_SEED)
    random_gain_rng=torch.Generator(device=device).manual_seed(RANDOM_GAIN_SEED)
    band_rng=torch.Generator(device=device).manual_seed(BAND_SEED)
    history=[];diagnostics=[];best_acer=float('inf');best_auc=-float('inf');best_epoch=0;no_improve=0;start=1
    if last_path.is_file():
        if not RESUME:raise RuntimeError(name+' has a saved state; enable RESUME or use a new output root')
        state=torch.load(last_path,map_location='cpu',weights_only=False)
        if state.get('run_id')!=name or state.get('config')!=cfg:raise RuntimeError(name+' resume run/config differs')
        if state['best_epoch']:
            if not best_path.is_file():raise RuntimeError(name+' missing best checkpoint on resume')
            old_best=torch.load(best_path,map_location='cpu',weights_only=True)
            if old_best['run_id']!=name or old_best['epoch']!=state['best_epoch'] or old_best['config']!=cfg:
                raise RuntimeError(name+' best checkpoint differs from resume state')
        model.load_state_dict(state['model']);optimizer.load_state_dict(state['optimizer'])
        scheduler.load_state_dict(state['scheduler']);scaler.load_state_dict(state['scaler'])
        history=state['history'];diagnostics=state['diagnostics'];best_acer=state['best_acer']
        best_auc=state['best_auc'];best_epoch=state['best_epoch'];no_improve=state['no_improve']
        start=state['epoch']+1
        if len(history)!=state['epoch']:raise RuntimeError(name+' resume history length differs')
        restore_rng(state['rng'],gain_rng,random_gain_rng,band_rng)
        print(name,'resume at epoch',start)
    val_dl=loader(CelebaData(val))
    masks=radial_masks()
    if start<=MAX_EPOCHS and not (history and history[-1]['stop_reached']):
        for epoch in range(start,MAX_EPOCHS+1):
            warmup=spec['method']!='CLEAN' and epoch<=WARMUP_EPOCHS
            active=not warmup
            model.train();totals=Counter();band_count=Counter();random_count=Counter();seen=0;harmful_count_total=0
            train_ds=CelebaData(train,training=True,epoch=epoch)
            for x,_,binary,spoof,lighting,attrs,_ in loader(train_ds,shuffle=True,epoch=epoch):
                x=x.to(device,non_blocking=True);binary=binary.to(device);spoof=spoof.to(device)
                lighting=lighting.to(device);attrs=attrs.to(device);optimizer.zero_grad(set_to_none=True)
                loss,scalars,bands,random_bands=training_terms(model,x,binary,spoof,lighting,attrs,
                    spec,masks,gain_rng,random_gain_rng,band_rng,active)
                scaler.scale(loss).backward();scaler.step(optimizer);scaler.update()
                count=len(x);seen+=count
                for key,value in scalars.items():totals[key]+=value*count
                band_count.update(bands);random_count.update(random_bands)
                harmful_count_total+=scalars['harmful_count']
            keys,y,z=infer(model,val_dl)
            if keys!=val.sample_key.astype(str).tolist():raise RuntimeError('Val order differs')
            scores=pad_score(z);threshold=threshold_from_val(y,scores)
            vm=metrics(y,scores,threshold,len(val))
            improved=active and (vm['acer']<best_acer-METRIC_TIE_EPS or
                (abs(vm['acer']-best_acer)<=METRIC_TIE_EPS and vm['auc']>best_auc+METRIC_TIE_EPS))
            if active:
                if improved:
                    best_acer=vm['acer'];best_auc=vm['auc'];best_epoch=epoch;no_improve=0
                    save_torch(best_path,{'run_id':name,'epoch':epoch,'config':cfg,
                        'val_acer':best_acer,'val_auc':best_auc,
                        'state_dict':{k:v.detach().cpu().clone() for k,v in model.state_dict().items()}})
                else:no_improve+=1
            row={'epoch':epoch,'phase':'warmup' if warmup else 'active','lr':optimizer.param_groups[0]['lr'],
                 'eligible_for_final_checkpoint':active,'val_acer':vm['acer'],'val_auc':vm['auc'],
                 'val_eer':vm['eer'],'val_threshold':threshold,'best_epoch':best_epoch,
                 'no_improve_count':no_improve,'clean_ce_weight':1.0 if warmup else spec['clean_ce_weight'],
                 'spectral_ce_weight':0.0 if warmup else spec['spectral_ce_weight'],
                 'consistency_beta':0.0 if warmup else spec['consistency_beta'],
                 'method':'CLEAN_WARMUP' if warmup else spec['method'],
                 **{k:v/seen for k,v in totals.items()},
                 **{f'{label}_count':band_count[i] for i,label in enumerate(('low','mid','high'))},
                 **{f'{label}_fraction':band_count[i]/seen for i,label in enumerate(('low','mid','high'))},
                 **{f'random_{label}_count':random_count[i] for i,label in enumerate(('low','mid','high'))},
                 **{f'random_{label}_fraction':random_count[i]/seen for i,label in enumerate(('low','mid','high'))}}
            row['harmful_count']=harmful_count_total
            if active and spec['method']!='CLEAN':diagnostics.append(dict(row))
            stop=epoch==MAX_EPOCHS or (active and epoch>=EARLIEST_EARLY_STOP_EPOCH and no_improve>=EARLY_STOP_PATIENCE)
            row['stop_reached']=stop;history.append(row)
            scheduler.step()
            state={'run_id':name,'config':cfg,'epoch':epoch,
                'model':{k:v.detach().cpu().clone() for k,v in model.state_dict().items()},
                'optimizer':optimizer.state_dict(),'scheduler':scheduler.state_dict(),
                'scaler':scaler.state_dict(),'best_epoch':best_epoch,'best_acer':best_acer,'best_auc':best_auc,
                'no_improve':no_improve,'rng':snapshot_rng(gain_rng,random_gain_rng,band_rng),
                'history':history,'diagnostics':diagnostics}
            save_torch(last_path,state)
            pd.DataFrame(history).to_csv(folder/'training_history.csv',index=False)
            if spec['method']!='CLEAN':pd.DataFrame(diagnostics).to_csv(folder/'spectral_diagnostics.csv',index=False)
            print(name,'epoch',epoch,'Val ACER',vm['acer'],'best',best_epoch,'patience',no_improve,flush=True)
            if stop:break
    if not best_path.is_file() or best_epoch<= (0 if spec['method']=='CLEAN' else WARMUP_EPOCHS):
        raise RuntimeError(name+' has no eligible best checkpoint')
    best=torch.load(best_path,map_location='cpu',weights_only=True)
    if best['run_id']!=name or best['config']!=cfg or best['epoch']!=best_epoch:
        raise RuntimeError(name+' best checkpoint identity differs')
    model.load_state_dict(best['state_dict']);model.eval()
    keys,y,z=infer(model,val_dl)
    if keys!=val.sample_key.astype(str).tolist():raise RuntimeError('Frozen Val order differs')
    locked=threshold_from_val(y,pad_score(z))
    vm=metrics(y,pad_score(z),locked,len(val))
    if abs(vm['acer']-best['val_acer'])>1e-6 or abs(vm['auc']-best['val_auc'])>1e-6:
        raise RuntimeError('Best checkpoint does not reproduce Val selection metrics')
    write_json(folder/'val_metrics.json',vm)
    prediction_frame(val,keys,y,z,locked,'sample_key').to_csv(folder/'val_predictions.csv',index=False)
    freeze={'run_id':name,'best_epoch':best_epoch,'locked_threshold':locked,'val_metrics':vm}
    freeze_path=folder/'evaluation_freeze.json'
    if freeze_path.is_file() and read_json(freeze_path)!=freeze:raise RuntimeError('Evaluation freeze differs')
    write_json(freeze_path,freeze)
    if EVAL_CELEBA_TEST:evaluate_celeba_test(model,folder,locked)
    evaluate_lcc(model,folder,locked)
    write_json(done_path,{'run_id':name,'best_epoch':best_epoch,'locked_threshold':locked,
                          'source_guard':'pending_summary'})
    del model
    torch.cuda.empty_cache()


In [ ]:
REPORT_NAME='micro_R6_R9_report.md'
REPORT_KIND='B'
selected=[name for name in RUNS if RUN_FILTER is None or name in RUN_FILTER]
for name in selected:
    try:run_one(name,RUNS[name])
    except torch.cuda.OutOfMemoryError as exc:
        raise RuntimeError('Frozen batch 256 ran out of GPU memory; stop this run') from exc

rows=[]
for name,spec in RUNS.items():
    folder=out/'runs'/name
    if not (folder/'completion.json').is_file():continue
    row={'run_id':name,'method':spec['method'],'clean_ce_weight':spec['clean_ce_weight'],
         'spectral_ce_weight':spec['spectral_ce_weight'],'consistency_beta':spec['consistency_beta'],
         'selected_epoch':read_json(folder/'completion.json')['best_epoch']}
    for split in ('val','lcc_training','lcc_development','lcc_evaluation','lcc_combined'):
        mm=read_json(folder/(split+'_metrics.json'))
        for key,value in mm.items():row[split+'_'+key]=value
    if EVAL_CELEBA_TEST:
        mm=read_json(folder/'test_metrics.json')
        for key,value in mm.items():row['test_'+key]=value
    rows.append(row)
ref=next((row['val_acer'] for row in rows if row['run_id']=='R0_clean'),None)
for row in rows:
    row['source_guard_pass']=(row['val_acer']<=ref+.005) if ref is not None else None
write_json(out/'comparison_summary.json',rows)
pd.DataFrame(rows).to_csv(out/'comparison_summary.csv',index=False)
report=['# Focused micro mechanism round','',
        'Single seed 100; frozen Train10K/Val3K; LCC is an external-development domain.',
        'CelebA full Test evaluation skipped in this micro screening round.' if not EVAL_CELEBA_TEST else
        'Optional CelebA full Test evaluation enabled after the Val3K freeze.',
        'No automatic winner is chosen. Source guard is descriptive.','',
        '| Run | Val ACER | Val AUC | LCC eval AUC | LCC eval EER | LCC eval HTER | LCC combined AUC | LCC combined EER | LCC combined HTER | LCC combined TPR@FPR1% | Guard |',
        '|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---|']
for row in rows:
    report.append(f"| {row['run_id']} | {row['val_acer']:.6f} | {row['val_auc']:.6f} | {row['lcc_evaluation_auc']:.6f} | {row['lcc_evaluation_eer']:.6f} | {row['lcc_evaluation_hter']:.6f} | {row['lcc_combined_auc']:.6f} | {row['lcc_combined_eer']:.6f} | {row['lcc_combined_hter']:.6f} | {row['lcc_combined_tpr_at_fpr_1pct']:.6f} | {row['source_guard_pass']} |")
if EVAL_CELEBA_TEST:
    report+=['','## Optional CelebA full Test','',
             '| Run | AUC | EER | HTER |','|---|---:|---:|---:|']
    for row in rows:
        report.append(f"| {row['run_id']} | {row['test_auc']:.6f} | {row['test_eer']:.6f} | {row['test_hter']:.6f} |")
report+=['','Detailed split metrics, candidate counts, scored counts and coverage are in `comparison_summary.csv` and each run’s metric JSON.',
         'Missing runs remain pending; compare the two notebooks manually after both finish.']
lookup={row['run_id']:row for row in rows}
report+=['','## Planned within-notebook comparisons','']
if REPORT_KIND=='A':
    for left,right in [('R0_clean','R1_p3_ce25'),('R1_p3_ce25','R2_worst_ce15'),
                       ('R1_p3_ce25','R3_worst_ce35'),('R1_p3_ce25','R4_worst_ce50')]:
        if left in lookup and right in lookup:
            report.append(f"- {right} minus {left}: Val ACER {lookup[right]['val_acer']-lookup[left]['val_acer']:+.6f}; LCC pooled AUC {lookup[right]['lcc_combined_auc']-lookup[left]['lcc_combined_auc']:+.6f}; LCC pooled HTER {lookup[right]['lcc_combined_hter']-lookup[left]['lcc_combined_hter']:+.6f}.")
        else:report.append(f'- {right} versus {left}: pending one or both runs.')
else:
    report.append('- R6 versus the P3 reference: pending manual merge with notebook A.')
    for name,columns in [('R7_harmful_gated_worst',['harmful_count','harmful_fraction','harmful_ce']),
                         ('R8_worst_random_mix',['random_low_fraction','random_mid_fraction','random_high_fraction']),
                         ('R9_worst_ce25_kl002',['kl_loss','weighted_kl_contribution'])]:
        path=out/'runs'/name/'spectral_diagnostics.csv'
        if path.is_file():
            frame=pd.read_csv(path)
            report.append('- '+name+': '+', '.join(f'{c}={frame[c].mean():.6f}' for c in columns)+'.')
        else:report.append('- '+name+': pending.')
report+=['','This single-seed micro round is descriptive. No method is selected automatically.']
(out/REPORT_NAME).write_text('\n'.join(report)+'\n')
zip_path=out.parent/(out.name+'.zip')
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED,allowZip64=True) as archive:
    for path in sorted(out.rglob('*')):
        if path.is_file():archive.write(path,(Path(out.name)/path.relative_to(out)).as_posix())
print('Completed',len(rows),'of',len(RUNS),'runs')
display(FileLink(str(zip_path)))
